# Analyse détaillée de la volumétrie avec DuckDB

Ce notebook :
- convertit le fichier CSV en Parquet si besoin ;
- affiche un aperçu des données ;
- calcule la volumétrie globale ;
- mesure les **nulls par colonne** ;
- mesure les **doublons exacts de lignes** ;
- produit une **analyse détaillée par colonne** ;
- exporte les résultats en CSV.

Le notebook est écrit pour éviter les requêtes SQL codées en dur sur chaque colonne.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# Fichiers d'entrée / sortie
csv_file = Path("A202505.csv")
parquet_file = Path("A202505.parquet")

# Paramètres CSV : adapte 'delim' si besoin
csv_delim = ";"
csv_header = True

con = duckdb.connect()

print("DuckDB version :", duckdb.__version__)
print("CSV attendu :", csv_file.resolve())
print("Parquet attendu :", parquet_file.resolve())

DuckDB version : 1.5.1
CSV attendu : /Users/la_brioche/kDrive/Master M1 Drive/S2/Projet FIN/A202505.csv
Parquet attendu : /Users/la_brioche/kDrive/Master M1 Drive/S2/Projet FIN/A202505.parquet


## 1. Conversion CSV → Parquet

Cette cellule ne reconvertit pas le fichier si le Parquet existe déjà.

In [2]:
if parquet_file.exists():
    print(f"Le fichier Parquet existe déjà : {parquet_file}")
else:
    if not csv_file.exists():
        raise FileNotFoundError(f"Fichier introuvable : {csv_file}")

    con.execute(f'''
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{csv_file.as_posix()}',
                delim='{csv_delim}',
                header={str(csv_header).lower()}
            )
        )
        TO '{parquet_file.as_posix()}' (FORMAT PARQUET)
    ''')
    print(f"Conversion terminée : {parquet_file}")

Le fichier Parquet existe déjà : A202505.parquet


## 2. Aperçu rapide

In [3]:
preview_df = con.execute(f"SELECT * FROM '{parquet_file.as_posix()}' LIMIT 10").df()
preview_df

,FLX_ANN_MOI,ORG_CLE_REG,AGE_BEN_SNDS,BEN_RES_REG,BEN_CMU_TOP,BEN_QLT_COD,BEN_SEX_COD,DDP_SPE_COD,ETE_CAT_SNDS,ETE_REG_COD,...,PSE_SPE_SNDS,PSE_STJ_SNDS,PRE_INS_REG,PSP_ACT_SNDS,PSP_ACT_CAT,PSP_SPE_SNDS,PSP_STJ_SNDS,TOP_PS5_TRG,ETB_DCS_MCO,column56
0,202505,76,70,76,1,1,2,0,1101,76,...,0,2,99,0,0,1,2,1,M,None
1,202505,11,70,11,0,1,1,121,9999,99,...,0,1,11,0,1,1,1,1,Z,None
2,202505,11,30,11,0,1,2,45,1107,11,...,0,2,99,0,99,0,9,1,S,None
3,202505,52,50,52,0,1,2,121,9999,99,...,0,1,99,0,0,42,2,1,Z,None
4,202505,11,80,11,1,1,2,121,9999,99,...,0,1,99,0,0,1,2,1,Z,None
5,202505,76,70,76,0,1,1,121,9999,99,...,0,1,76,0,1,1,1,1,Z,None
6,202505,11,70,11,0,1,2,0,1102,11,...,4,2,99,0,0,1,2,1,C,None
7,202505,32,80,32,0,1,2,121,9999,99,...,1,1,32,0,1,1,1,1,Z,None
8,202505,99,80,99,0,1,2,121,9999,99,...,0,1,99,0,0,31,2,1,Z,None
9,202505,11,80,76,0,1,2,121,9999,99,...,0,1,76,0,1,12,1,1,Z,None


In [4]:
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{parquet_file.as_posix()}'").df()
schema_df

,column_name,column_type,null,key,default,extra
0,FLX_ANN_MOI,BIGINT,YES,None,None,None
1,ORG_CLE_REG,BIGINT,YES,None,None,None
2,AGE_BEN_SNDS,BIGINT,YES,None,None,None
3,BEN_RES_REG,BIGINT,YES,None,None,None
4,BEN_CMU_TOP,BIGINT,YES,None,None,None
5,BEN_QLT_COD,BIGINT,YES,None,None,None
6,BEN_SEX_COD,BIGINT,YES,None,None,None
7,DDP_SPE_COD,BIGINT,YES,None,None,None
8,ETE_CAT_SNDS,BIGINT,YES,None,None,None
9,ETE_REG_COD,BIGINT,YES,None,None,None


## 3. Métadonnées globales

On récupère :
- le nombre de lignes ;
- le nombre de colonnes ;
- le nombre total de cellules ;
- le nombre total de cellules nulles ;
- le pourcentage global de nulls.

In [5]:
table_name = "volumetrie_tmp"

con.execute(f"CREATE OR REPLACE TEMP VIEW {table_name} AS SELECT * FROM '{parquet_file.as_posix()}'")

columns_df = con.execute(f"DESCRIBE SELECT * FROM {table_name}").df()
columns = columns_df["column_name"].tolist()
column_types = dict(zip(columns_df["column_name"], columns_df["column_type"]))

nb_colonnes = len(columns)
nb_lignes = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

null_exprs = [f"SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}" for col in columns]
nulls_wide_df = con.execute(f"SELECT {', '.join(null_exprs)} FROM {table_name}").df()

nulls_long_df = (
    nulls_wide_df.T
    .reset_index()
    .rename(columns={"index": "column_name", 0: "null_count"})
)
nulls_long_df["null_count"] = nulls_long_df["null_count"].astype("int64")
nulls_long_df["null_pct"] = (nulls_long_df["null_count"] / nb_lignes * 100).round(4)
nulls_long_df["column_type"] = nulls_long_df["column_name"].map(column_types)
nulls_long_df = nulls_long_df[["column_name", "column_type", "null_count", "null_pct"]].sort_values(
    by=["null_count", "column_name"], ascending=[False, True]
).reset_index(drop=True)

nb_null_total = int(nulls_long_df["null_count"].sum())
nb_cellules = int(nb_lignes * nb_colonnes)
pct_null_global = round((nb_null_total / nb_cellules * 100), 4) if nb_cellules else 0.0

global_df = pd.DataFrame([{
    "nb_lignes": nb_lignes,
    "nb_colonnes": nb_colonnes,
    "nb_cellules_total": nb_cellules,
    "nb_null_total": nb_null_total,
    "pct_null_global": pct_null_global
}])

global_df

,nb_lignes,nb_colonnes,nb_cellules_total,nb_null_total,pct_null_global
0,34167564,57,1947551148,42076150,2.1605


## 4. Doublons exacts de lignes

Ici, un doublon signifie : **une ligne strictement identique sur toutes les colonnes**.
- `nb_groupes_doublons` = nombre de groupes ayant au moins 2 lignes identiques ;
- `nb_lignes_dupliquees` = nombre de lignes en trop par rapport à une occurrence unique.

In [6]:
duplicates_df = con.execute(f'''
WITH groupes AS (
    SELECT COUNT(*) AS cnt
    FROM {table_name}
    GROUP BY ALL
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS nb_groupes_doublons,
    COALESCE(SUM(cnt - 1), 0) AS nb_lignes_dupliquees,
    COALESCE(SUM(cnt), 0) AS nb_lignes_concernees
FROM groupes
''').df()

duplicates_df

,nb_groupes_doublons,nb_lignes_dupliquees,nb_lignes_concernees
0,1,34167563.0,34167564.0


## 5. Nulls par colonne

In [7]:
nulls_long_df

,column_name,column_type,null_count,null_pct
0,column56,VARCHAR,34167564,100.0000
1,PRS_ACT_NBR,BIGINT,4120235,12.0589
2,FLT_ACT_NBR,BIGINT,3788351,11.0876
3,AGE_BEN_SNDS,BIGINT,0,0.0000
4,ASU_NAT,BIGINT,0,0.0000
5,ATT_NAT,BIGINT,0,0.0000
6,BEN_CMU_TOP,BIGINT,0,0.0000
7,BEN_QLT_COD,BIGINT,0,0.0000
8,BEN_RES_REG,BIGINT,0,0.0000
9,BEN_SEX_COD,BIGINT,0,0.0000


## 6. Analyse détaillée par colonne

Pour chaque colonne :
- type ;
- nombre de valeurs non nulles ;
- nombre de nulls ;
- pourcentage de nulls ;
- nombre de valeurs distinctes non nulles.

Attention : `COUNT(DISTINCT ...)` peut être coûteux sur un très gros fichier.

In [8]:
detail_rows = []

for col in columns:
    q = f'''
    SELECT
        COUNT(*) AS row_count,
        COUNT("{col}") AS non_null_count,
        SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS null_count,
        COUNT(DISTINCT "{col}") AS distinct_non_null_count
    FROM {table_name}
    '''
    row = con.execute(q).fetchone()
    row_count, non_null_count, null_count, distinct_non_null_count = row
    detail_rows.append({
        "column_name": col,
        "column_type": column_types[col],
        "row_count": int(row_count),
        "non_null_count": int(non_null_count),
        "null_count": int(null_count),
        "null_pct": round((null_count / row_count * 100), 4) if row_count else 0.0,
        "distinct_non_null_count": int(distinct_non_null_count)
    })

detail_df = pd.DataFrame(detail_rows).sort_values(
    by=["null_count", "column_name"], ascending=[False, True]
).reset_index(drop=True)

detail_df

,column_name,column_type,row_count,non_null_count,null_count,null_pct,distinct_non_null_count
0,column56,VARCHAR,34167564,0,34167564,100.0000,0
1,PRS_ACT_NBR,BIGINT,34167564,30047329,4120235,12.0589,18392
2,FLT_ACT_NBR,BIGINT,34167564,30379213,3788351,11.0876,17946
3,AGE_BEN_SNDS,BIGINT,34167564,34167564,0,0.0000,9
4,ASU_NAT,BIGINT,34167564,34167564,0,0.0000,11
5,ATT_NAT,BIGINT,34167564,34167564,0,0.0000,4
6,BEN_CMU_TOP,BIGINT,34167564,34167564,0,0.0000,4
7,BEN_QLT_COD,BIGINT,34167564,34167564,0,0.0000,5
8,BEN_RES_REG,BIGINT,34167564,34167564,0,0.0000,14
9,BEN_SEX_COD,BIGINT,34167564,34167564,0,0.0000,3


## 7. Résumé final consolidé

In [9]:
resume_df = global_df.copy()
resume_df["nb_groupes_doublons"] = int(duplicates_df.loc[0, "nb_groupes_doublons"])
resume_df["nb_lignes_dupliquees"] = int(duplicates_df.loc[0, "nb_lignes_dupliquees"])
resume_df["nb_lignes_concernees_par_doublons"] = int(duplicates_df.loc[0, "nb_lignes_concernees"])

resume_df

,nb_lignes,nb_colonnes,nb_cellules_total,nb_null_total,pct_null_global,nb_groupes_doublons,nb_lignes_dupliquees,nb_lignes_concernees_par_doublons
0,34167564,57,1947551148,42076150,2.1605,1,34167563,34167564
